# Aviation Delay Research — Data Ingestion Pipeline

**Goal:** Fetch, clean, and store historical US flight delay data and hourly weather observations for later regression analysis.

| | |
|---|---|
| **Airports** | Top 50 busiest US airports |
| **Period** | January 2022 – December 2024 |
| **Granularity** | Airport × UTC hour |
| **Sources** | FAA BTS On-Time Performance · Meteostat |
| **Storage** | DuckDB + Parquet |

---

## Table of contents
1. [Setup & configuration](#1-setup)
2. [Airport list](#2-airports)
3. [Flight data ingestion (BTS)](#3-flights)
4. [Weather data ingestion (Meteostat)](#4-weather)
5. [Cleaning & standardisation](#5-cleaning)
6. [Airport-hour panel dataset](#6-panel)
7. [Exports](#7-exports)
8. [Quick data inspection](#8-inspect)
9. [Descriptive statistics](#9-stats)

## 0 · Install dependencies
Uncomment and run once if packages are not already installed.

In [10]:
%pip install -q pandas pyarrow duckdb "meteostat>=2.1" requests tqdm polars

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


<a id='1-setup'></a>
## 1 · Setup & configuration

In [11]:
import io
import logging
import time
import warnings
import zipfile
from datetime import datetime
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from meteostat import hourly, Point, stations   # v2 API: lowercase functions

warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


In [12]:
# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT            = Path('.').resolve()
DATA_DIR        = ROOT / 'data'
RAW_FLIGHTS_DIR = DATA_DIR / 'raw' / 'flights'
RAW_WEATHER_DIR = DATA_DIR / 'raw' / 'weather'
PROCESSED_DIR   = DATA_DIR / 'processed'
EXPORTS_DIR     = DATA_DIR / 'exports'
LOGS_DIR        = ROOT / 'logs'
SCHEMA_DIR      = ROOT / 'schema'
DB_PATH         = DATA_DIR / 'aviation.duckdb'

for d in [RAW_FLIGHTS_DIR, RAW_WEATHER_DIR, PROCESSED_DIR, EXPORTS_DIR, LOGS_DIR, SCHEMA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Study period ──────────────────────────────────────────────────────────────
START_YEAR, START_MONTH = 2022, 1
END_YEAR,   END_MONTH   = 2024, 12

# ── BTS settings ─────────────────────────────────────────────────────────────
BTS_BASE_URL = (
    'https://transtats.bts.gov/PREZIP/'
    'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip'
)
BTS_COLUMNS = [
    'FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline',
    'Tail_Number', 'Origin', 'Dest',
    'CRSDepTime', 'DepTime', 'DepDelay',
    'CRSArrTime', 'ArrTime', 'ArrDelay',
    'Cancelled', 'CancellationCode',
]
BTS_RENAME = {
    'FlightDate':                      'flight_date',
    'Reporting_Airline':               'carrier',
    'Flight_Number_Reporting_Airline': 'flight_number',
    'Tail_Number':                     'tail_number',
    'Origin':                          'origin_airport',
    'Dest':                            'destination_airport',
    'CRSDepTime':                      'dep_time_scheduled',
    'DepTime':                         'dep_time_actual',
    'DepDelay':                        'dep_delay',
    'CRSArrTime':                      'arr_time_scheduled',
    'ArrTime':                         'arr_time_actual',
    'ArrDelay':                        'arr_delay',
    'Cancelled':                       'cancelled',
    'CancellationCode':                'cancellation_code',
}

# ── Processing settings ───────────────────────────────────────────────────────
CHUNK_SIZE               = 100_000
PARQUET_COMPRESSION      = 'snappy'
DELAY_MIN                = -120
DELAY_MAX                = 1_440
STATION_SEARCH_RADIUS_KM = 50
REQUEST_DELAY            = 2.0    # seconds between BTS requests
MAX_RETRIES              = 3

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOGS_DIR / 'pipeline.log', encoding='utf-8'),
    ],
)
log = logging.getLogger('notebook')
log.info('Configuration loaded. ROOT=%s', ROOT)

2026-05-08 12:48:45,672 [INFO] Configuration loaded. ROOT=C:\Fun Projects\WDI - Jupyter Project


### DuckDB helpers (shared throughout notebook)

In [13]:
def get_con(read_only: bool = False) -> duckdb.DuckDBPyConnection:
    """Open the project DuckDB database."""
    con = duckdb.connect(str(DB_PATH), read_only=read_only)
    con.execute("PRAGMA threads=4")
    con.execute("PRAGMA memory_limit='2GB'")
    return con


def write_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(
        pa.Table.from_pandas(df, preserve_index=False),
        str(path),
        compression=PARQUET_COMPRESSION,
        use_dictionary=True,
    )


def append_to_table(df: pd.DataFrame, table: str) -> int:
    if df.empty:
        return 0
    con = get_con()
    con.register('_tmp', df)
    con.execute(f'INSERT INTO {table} SELECT * FROM _tmp')
    con.unregister('_tmp')
    con.close()
    return len(df)


def query(sql: str) -> pd.DataFrame:
    con = get_con(read_only=True)
    df  = con.execute(sql).df()
    con.close()
    return df


print('DuckDB helpers defined')

DuckDB helpers defined


<a id='2-airports'></a>
## 2 · Airport list

Top 50 busiest US airports by enplanements (FAA 2019–2023).  
Stored in-notebook — no external download required for this step.

In [14]:
AIRPORT_ROWS = [
    ('ATL','KATL','Hartsfield-Jackson Atlanta International',     'Atlanta',       'GA', 33.6367, -84.4281, 'America/New_York'),
    ('LAX','KLAX','Los Angeles International',                    'Los Angeles',   'CA', 33.9425,-118.4081, 'America/Los_Angeles'),
    ('ORD','KORD',"O'Hare International",                        'Chicago',       'IL', 41.9742, -87.9073, 'America/Chicago'),
    ('DFW','KDFW','Dallas/Fort Worth International',              'Dallas',        'TX', 32.8998, -97.0403, 'America/Chicago'),
    ('DEN','KDEN','Denver International',                         'Denver',        'CO', 39.8561,-104.6737, 'America/Denver'),
    ('JFK','KJFK','John F. Kennedy International',                'New York',      'NY', 40.6413, -73.7781, 'America/New_York'),
    ('SFO','KSFO','San Francisco International',                  'San Francisco', 'CA', 37.6213,-122.3790, 'America/Los_Angeles'),
    ('SEA','KSEA','Seattle-Tacoma International',                 'Seattle',       'WA', 47.4502,-122.3088, 'America/Los_Angeles'),
    ('LAS','KLAS','Harry Reid International',                     'Las Vegas',     'NV', 36.0840,-115.1537, 'America/Los_Angeles'),
    ('MCO','KMCO','Orlando International',                        'Orlando',       'FL', 28.4294, -81.3089, 'America/New_York'),
    ('EWR','KEWR','Newark Liberty International',                 'Newark',        'NJ', 40.6895, -74.1745, 'America/New_York'),
    ('MIA','KMIA','Miami International',                          'Miami',         'FL', 25.7959, -80.2870, 'America/New_York'),
    ('PHX','KPHX','Phoenix Sky Harbor International',             'Phoenix',       'AZ', 33.4373,-112.0078, 'America/Phoenix'),
    ('IAH','KIAH','George Bush Intercontinental',                 'Houston',       'TX', 29.9902, -95.3368, 'America/Chicago'),
    ('BOS','KBOS','Boston Logan International',                   'Boston',        'MA', 42.3656, -71.0096, 'America/New_York'),
    ('MSP','KMSP','Minneapolis-Saint Paul International',         'Minneapolis',   'MN', 44.8848, -93.2223, 'America/Chicago'),
    ('DTW','KDTW','Detroit Metropolitan Wayne County',            'Detroit',       'MI', 42.2162, -83.3554, 'America/Detroit'),
    ('FLL','KFLL','Fort Lauderdale-Hollywood International',      'Fort Lauderdale','FL',26.0726, -80.1527, 'America/New_York'),
    ('PHL','KPHL','Philadelphia International',                   'Philadelphia',  'PA', 39.8744, -75.2424, 'America/New_York'),
    ('LGA','KLGA','LaGuardia',                                    'New York',      'NY', 40.7772, -73.8726, 'America/New_York'),
    ('BWI','KBWI','Baltimore/Washington International',           'Baltimore',     'MD', 39.1754, -76.6683, 'America/New_York'),
    ('DCA','KDCA','Ronald Reagan Washington National',            'Washington',    'DC', 38.8512, -77.0402, 'America/New_York'),
    ('IAD','KIAD','Washington Dulles International',              'Dulles',        'VA', 38.9531, -77.4565, 'America/New_York'),
    ('MDW','KMDW','Chicago Midway International',                 'Chicago',       'IL', 41.7868, -87.7522, 'America/Chicago'),
    ('SLC','KSLC','Salt Lake City International',                 'Salt Lake City','UT', 40.7884,-111.9778, 'America/Denver'),
    ('SAN','KSAN','San Diego International',                      'San Diego',     'CA', 32.7338,-117.1933, 'America/Los_Angeles'),
    ('TPA','KTPA','Tampa International',                          'Tampa',         'FL', 27.9755, -82.5332, 'America/New_York'),
    ('PDX','KPDX','Portland International',                       'Portland',      'OR', 45.5898,-122.5951, 'America/Los_Angeles'),
    ('HNL','PHNL','Daniel K. Inouye International',               'Honolulu',      'HI', 21.3187,-157.9225, 'Pacific/Honolulu'),
    ('STL','KSTL','St. Louis Lambert International',              'St. Louis',     'MO', 38.7487, -90.3700, 'America/Chicago'),
    ('BNA','KBNA','Nashville International',                      'Nashville',     'TN', 36.1245, -86.6782, 'America/Chicago'),
    ('AUS','KAUS','Austin-Bergstrom International',               'Austin',        'TX', 30.1975, -97.6664, 'America/Chicago'),
    ('DAL','KDAL','Dallas Love Field',                            'Dallas',        'TX', 32.8471, -96.8517, 'America/Chicago'),
    ('HOU','KHOU','William P. Hobby',                             'Houston',       'TX', 29.6454, -95.2789, 'America/Chicago'),
    ('MSY','KMSY','Louis Armstrong New Orleans International',    'New Orleans',   'LA', 29.9934, -90.2580, 'America/Chicago'),
    ('RDU','KRDU','Raleigh-Durham International',                 'Raleigh',       'NC', 35.8776, -78.7875, 'America/New_York'),
    ('CLT','KCLT','Charlotte Douglas International',              'Charlotte',     'NC', 35.2140, -80.9431, 'America/New_York'),
    ('MEM','KMEM','Memphis International',                        'Memphis',       'TN', 35.0424, -89.9767, 'America/Chicago'),
    ('OAK','KOAK','Oakland International',                        'Oakland',       'CA', 37.7213,-122.2208, 'America/Los_Angeles'),
    ('SJC','KSJC','Norman Y. Mineta San Jose International',      'San Jose',      'CA', 37.3626,-121.9290, 'America/Los_Angeles'),
    ('SAT','KSAT','San Antonio International',                    'San Antonio',   'TX', 29.5337, -98.4698, 'America/Chicago'),
    ('SMF','KSMF','Sacramento International',                     'Sacramento',    'CA', 38.6954,-121.5908, 'America/Los_Angeles'),
    ('MCI','KMCI','Kansas City International',                    'Kansas City',   'MO', 39.2976, -94.7139, 'America/Chicago'),
    ('OGG','PHOG','Kahului',                                      'Kahului',       'HI', 20.8986,-156.4305, 'Pacific/Honolulu'),
    ('ANC','PANC','Ted Stevens Anchorage International',          'Anchorage',     'AK', 61.1744,-149.9964, 'America/Anchorage'),
    ('IND','KIND','Indianapolis International',                   'Indianapolis',  'IN', 39.7173, -86.2944, 'America/Indiana/Indianapolis'),
    ('CMH','KCMH','John Glenn Columbus International',            'Columbus',      'OH', 39.9980, -82.8919, 'America/New_York'),
    ('PIT','KPIT','Pittsburgh International',                     'Pittsburgh',    'PA', 40.4915, -80.2329, 'America/New_York'),
    ('BDL','KBDL','Bradley International',                        'Windsor Locks', 'CT', 41.9389, -72.6832, 'America/New_York'),
    ('CLE','KCLE','Cleveland Hopkins International',              'Cleveland',     'OH', 41.4117, -81.8498, 'America/New_York'),
]

airports_df = pd.DataFrame(
    AIRPORT_ROWS,
    columns=['iata','icao','airport_name','city','state','latitude','longitude','timezone']
)
IATA_SET = set(airports_df['iata'])

print(f'{len(airports_df)} airports loaded')
airports_df.head()

50 airports loaded


,iata,icao,airport_name,city,state,latitude,longitude,timezone
0,ATL,KATL,Hartsfield-Jackson Atlanta International,Atlanta,GA,33.6367,-84.4281,America/New_York
1,LAX,KLAX,Los Angeles International,Los Angeles,CA,33.9425,-118.4081,America/Los_Angeles
2,ORD,KORD,O'Hare International,Chicago,IL,41.9742,-87.9073,America/Chicago
3,DFW,KDFW,Dallas/Fort Worth International,Dallas,TX,32.8998,-97.0403,America/Chicago
4,DEN,KDEN,Denver International,Denver,CO,39.8561,-104.6737,America/Denver


In [15]:
# ── Create DuckDB schema & load airports table ────────────────────────────────
con = get_con()

con.execute("""
    CREATE TABLE IF NOT EXISTS airports (
        iata         VARCHAR PRIMARY KEY,
        icao         VARCHAR,
        airport_name VARCHAR,
        city         VARCHAR,
        state        VARCHAR,
        latitude     DOUBLE,
        longitude    DOUBLE,
        timezone     VARCHAR
    )
""")

con.execute("""
    CREATE TABLE IF NOT EXISTS flights_raw (
        flight_date          DATE,
        carrier              VARCHAR,
        flight_number        VARCHAR,
        tail_number          VARCHAR,
        origin_airport       VARCHAR,
        destination_airport  VARCHAR,
        dep_time_scheduled   VARCHAR,
        dep_time_actual      VARCHAR,
        dep_delay            DOUBLE,
        arr_time_scheduled   VARCHAR,
        arr_time_actual      VARCHAR,
        arr_delay            DOUBLE,
        cancelled            BOOLEAN,
        cancellation_code    VARCHAR,
        source_file          VARCHAR
    )
""")

con.execute("""
    CREATE TABLE IF NOT EXISTS weather_raw (
        airport_iata  VARCHAR,
        station_id    VARCHAR,
        timestamp_utc TIMESTAMPTZ,
        temperature   DOUBLE,
        precipitation DOUBLE,
        snowfall      DOUBLE,
        wind_speed    DOUBLE,
        visibility    DOUBLE,
        pressure      DOUBLE
    )
""")

con.execute("""
    CREATE TABLE IF NOT EXISTS airport_hour_panel (
        airport_iata       VARCHAR,
        timestamp_utc      TIMESTAMPTZ,
        hour_of_day        INTEGER,
        day_of_week        INTEGER,
        month              INTEGER,
        year               INTEGER,
        weekend_indicator  BOOLEAN,
        total_departures   INTEGER,
        total_arrivals     INTEGER,
        cancellation_count INTEGER,
        avg_dep_delay      DOUBLE,
        avg_arr_delay      DOUBLE,
        avg_temperature    DOUBLE,
        avg_precipitation  DOUBLE,
        avg_snowfall       DOUBLE,
        avg_wind_speed     DOUBLE,
        avg_visibility     DOUBLE,
        avg_pressure       DOUBLE,
        PRIMARY KEY (airport_iata, timestamp_utc)
    )
""")

# Populate airports (idempotent)
con.execute('DELETE FROM airports')
con.register('airports_df', airports_df)
con.execute('INSERT INTO airports SELECT * FROM airports_df')
con.unregister('airports_df')
con.close()

print('Schema initialised. airports table populated.')
query('SELECT COUNT(*) AS n FROM airports')

Schema initialised. airports table populated.


,n
0,50


<a id='3-flights'></a>
## 3 · Flight data ingestion

Downloads FAA BTS On-Time Performance ZIP files **month by month**.  
Only flights involving a top-50 airport (origin **or** destination) are kept.  
Each month is cached as a Parquet file — re-running skips already-downloaded months.

> **Heads-up:** 36 months × ~10–30 MB/ZIP ≈ 30–90 minutes depending on your connection.

In [16]:
# ── Download helpers ──────────────────────────────────────────────────────────

def bts_url(year: int, month: int) -> str:
    return BTS_BASE_URL.format(year=year, month=month)


def flight_parquet_path(year: int, month: int) -> Path:
    return RAW_FLIGHTS_DIR / f'flights_{year}_{month:02d}.parquet'


def download_zip(url: str) -> bytes | None:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            log.info('GET %s (attempt %d)', url, attempt)
            resp = requests.get(url, timeout=120, stream=True)
            resp.raise_for_status()
            return resp.content
        except requests.RequestException as exc:
            log.warning('Attempt %d failed: %s', attempt, exc)
            if attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
    return None


def read_bts_csv(zip_bytes: bytes) -> pd.DataFrame | None:
    try:
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            csv_name = next((n for n in zf.namelist() if n.lower().endswith('.csv')), None)
            if not csv_name:
                return None
            with zf.open(csv_name) as fobj:
                available = pd.read_csv(fobj, nrows=0).columns.tolist()
                wanted = [c for c in BTS_COLUMNS if c in available]
                fobj.seek(0)
                chunks = [
                    chunk for chunk in
                    pd.read_csv(fobj, usecols=wanted, chunksize=CHUNK_SIZE,
                                dtype=str, low_memory=False)
                ]
        return pd.concat(chunks, ignore_index=True) if chunks else None
    except Exception as exc:
        log.error('ZIP parse error: %s', exc)
        return None


def filter_and_rename(df: pd.DataFrame, source: str) -> pd.DataFrame:
    rename = {k: v for k, v in BTS_RENAME.items() if k in df.columns}
    df = df.rename(columns=rename)

    mask = df['origin_airport'].isin(IATA_SET) | df['destination_airport'].isin(IATA_SET)
    df   = df[mask].copy()

    df['source_file'] = source

    for col in ('dep_delay', 'arr_delay'):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    if 'cancelled' in df.columns:
        df['cancelled'] = df['cancelled'].map(
            lambda x: True if str(x).strip() in ('1', '1.0', '1.00') else False
        )

    expected = list(BTS_RENAME.values()) + ['source_file']
    for col in expected:
        if col not in df.columns:
            df[col] = None
    return df[expected]


print('Flight ingestion helpers defined')

Flight ingestion helpers defined


In [17]:
# ── Download loop ─────────────────────────────────────────────────────────────
flight_summary = []

for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        if year == START_YEAR and month < START_MONTH:
            continue
        if year == END_YEAR and month > END_MONTH:
            continue

        out_path = flight_parquet_path(year, month)

        if out_path.exists() and out_path.stat().st_size > 0:
            cached_rows = len(pd.read_parquet(out_path))
            log.info('%d-%02d already cached (%d rows)', year, month, cached_rows)
            flight_summary.append({'year': year, 'month': month, 'rows': cached_rows, 'status': 'cached'})
            continue

        url       = bts_url(year, month)
        zip_bytes = download_zip(url)
        if not zip_bytes:
            flight_summary.append({'year': year, 'month': month, 'rows': 0, 'status': 'failed'})
            continue

        raw = read_bts_csv(zip_bytes)
        if raw is None or raw.empty:
            flight_summary.append({'year': year, 'month': month, 'rows': 0, 'status': 'empty'})
            continue

        filtered = filter_and_rename(raw, f'BTS_{year}_{month:02d}')
        write_parquet(filtered, out_path)
        log.info('%d-%02d: %d rows → %s', year, month, len(filtered), out_path.name)
        flight_summary.append({'year': year, 'month': month, 'rows': len(filtered), 'status': 'downloaded'})

        time.sleep(REQUEST_DELAY)

summary_df = pd.DataFrame(flight_summary)
print(f'\nTotal rows across all months: {summary_df["rows"].sum():,}')
summary_df

2026-05-08 12:48:45,774 [INFO] GET https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_1.zip (attempt 1)
2026-05-08 12:49:21,588 [INFO] 2022-01: 529088 rows → flights_2022_01.parquet
2026-05-08 12:49:23,589 [INFO] GET https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_2.zip (attempt 1)
2026-05-08 12:49:53,207 [INFO] 2022-02: 486972 rows → flights_2022_02.parquet
2026-05-08 12:49:55,208 [INFO] GET https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_3.zip (attempt 1)
2026-05-08 12:50:29,339 [INFO] 2022-03: 553653 rows → flights_2022_03.parquet
2026-05-08 12:50:31,341 [INFO] GET https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_4.zip (attempt 1)
2026-05-08 12:51:05,154 [INFO] 2022-04: 546081 rows → flights_2022_04.parquet
2026-05-08 12:51:07,155 [INFO] GET https://transtats.bts.gov/PREZIP/On_Time_Reporting_Ca


Total rows across all months: 20,334,570


,year,month,rows,status
0,2022,1,529088,downloaded
1,2022,2,486972,downloaded
2,2022,3,553653,downloaded
3,2022,4,546081,downloaded
4,2022,5,569486,downloaded
5,2022,6,567132,downloaded
6,2022,7,584297,downloaded
7,2022,8,581184,downloaded
8,2022,9,550456,downloaded
9,2022,10,564388,downloaded


In [18]:
# ── Load all flight Parquet files → DuckDB ────────────────────────────────────
con = get_con()
con.execute('DELETE FROM flights_raw')
con.close()

total_flight_rows = 0
for pf in sorted(RAW_FLIGHTS_DIR.glob('flights_*.parquet')):
    df   = pd.read_parquet(pf)
    rows = append_to_table(df, 'flights_raw')
    total_flight_rows += rows
    log.info('Loaded %s → %d rows', pf.name, rows)

print(f'flights_raw total: {total_flight_rows:,} rows')
query('SELECT COUNT(*) AS total_flights FROM flights_raw')

2026-05-08 13:13:47,067 [INFO] Loaded flights_2022_01.parquet → 529088 rows
2026-05-08 13:13:48,473 [INFO] Loaded flights_2022_02.parquet → 486972 rows
2026-05-08 13:13:49,997 [INFO] Loaded flights_2022_03.parquet → 553653 rows
2026-05-08 13:13:51,691 [INFO] Loaded flights_2022_04.parquet → 546081 rows
2026-05-08 13:13:53,306 [INFO] Loaded flights_2022_05.parquet → 569486 rows
2026-05-08 13:13:54,843 [INFO] Loaded flights_2022_06.parquet → 567132 rows
2026-05-08 13:13:56,503 [INFO] Loaded flights_2022_07.parquet → 584297 rows
2026-05-08 13:13:58,185 [INFO] Loaded flights_2022_08.parquet → 581184 rows
2026-05-08 13:13:59,762 [INFO] Loaded flights_2022_09.parquet → 550456 rows
2026-05-08 13:14:01,186 [INFO] Loaded flights_2022_10.parquet → 564388 rows
2026-05-08 13:14:02,561 [INFO] Loaded flights_2022_11.parquet → 538579 rows
2026-05-08 13:14:04,001 [INFO] Loaded flights_2022_12.parquet → 547803 rows
2026-05-08 13:14:05,401 [INFO] Loaded flights_2023_01.parquet → 530807 rows
2026-05-08 1

flights_raw total: 20,334,570 rows


,total_flights
0,20334570


<a id='4-weather'></a>
## 4 · Weather data ingestion

Fetches **hourly** surface observations from Meteostat for each airport using the `Point` API (nearest station within 50 km).  
Each airport is cached as a Parquet file — re-running skips airports already fetched.

In [ ]:
# ── Weather helpers (meteostat v2 API) ───────────────────────────────────────────
START_DT = datetime(START_YEAR, START_MONTH, 1)
END_DT   = datetime(END_YEAR,   END_MONTH,   31)

# v2 column names → our internal names
WEATHER_RENAME = {
    'temp': 'temperature',
    'prcp': 'precipitation',
    'snwd': 'snowfall',
    'wspd': 'wind_speed',
    'pres': 'pressure',
    'vsby': 'visibility',
}


def weather_parquet_path(iata: str) -> Path:
    return RAW_WEATHER_DIR / f'weather_{iata}.parquet'


def find_station_id(lat: float, lon: float) -> str:
    """Return nearest Meteostat station ID within search radius, or empty string."""
    try:
        result = stations.nearby(Point(lat, lon), radius=STATION_SEARCH_RADIUS_KM * 1000)
        return str(result.index[0]) if not result.empty else ''
    except Exception:
        return ''


def fetch_hourly_weather(lat: float, lon: float, iata: str) -> pd.DataFrame | None:
    """
    Fetch hourly weather via meteostat v2 station ID API.
    Returns a tidy DataFrame with standardised column names, or None on failure.
    """
    try:
        nearby = stations.nearby(Point(lat, lon), radius=STATION_SEARCH_RADIUS_KM * 1000)
        if nearby.empty:
            log.warning('%s: no nearby station found', iata)
            return None
        station_id = nearby.index[0]

        ts = hourly(station_id, START_DT, END_DT)
        df = ts.fetch()

        if df is None or df.empty:
            return None

        # v2: index is 'time'; reset to bring it into a column
        df = df.reset_index().rename(columns={'time': 'timestamp_utc'})

        # Keep only our target columns (some may be absent depending on station)
        keep = ['timestamp_utc'] + [c for c in WEATHER_RENAME if c in df.columns]
        df = df[keep].rename(columns=WEATHER_RENAME)

        # Ensure all required columns exist (fill absent ones with NaN)
        for col in ('temperature', 'precipitation', 'snowfall',
                    'wind_speed', 'visibility', 'pressure'):
            if col not in df.columns:
                df[col] = float('nan')

        df['airport_iata'] = iata
        df['station_id']   = str(station_id)

        final_cols = ['airport_iata', 'station_id', 'timestamp_utc',
                      'temperature', 'precipitation', 'snowfall',
                      'wind_speed', 'visibility', 'pressure']
        return df[final_cols]

    except Exception as exc:
        log.error('%s: weather fetch failed — %s', iata, exc)
        return None


print('Weather helpers defined (meteostat v2)')

In [ ]:
# ── Fetch loop ──────────────────────────────────────────────────────────────────────────────────
weather_summary = []

for _, ap in airports_df.iterrows():
    iata     = ap['iata']
    out_path = weather_parquet_path(iata)

    if out_path.exists() and out_path.stat().st_size > 0:
        n = len(pd.read_parquet(out_path))
        log.info('%s already cached (%d rows)', iata, n)
        weather_summary.append({'iata': iata, 'rows': n, 'status': 'cached'})
        continue

    log.info('Fetching weather for %s (%s)', iata, ap['airport_name'])
    df = fetch_hourly_weather(ap['latitude'], ap['longitude'], iata)

    if df is None or df.empty:
        weather_summary.append({'iata': iata, 'rows': 0, 'status': 'no_data'})
        continue

    write_parquet(df, out_path)
    log.info('%s: %d rows saved', iata, len(df))
    weather_summary.append({'iata': iata, 'rows': len(df), 'status': 'fetched'})

wx_summary_df = pd.DataFrame(weather_summary)
print(f'Total weather rows: {wx_summary_df["rows"].sum():,}')
wx_summary_df

In [21]:
# ── Load all weather Parquet → DuckDB ────────────────────────────────────────
con = get_con()
con.execute('DELETE FROM weather_raw')
con.close()

total_weather_rows = 0
for pf in sorted(RAW_WEATHER_DIR.glob('weather_*.parquet')):
    df   = pd.read_parquet(pf)
    rows = append_to_table(df, 'weather_raw')
    total_weather_rows += rows

print(f'weather_raw total: {total_weather_rows:,} rows')
query('SELECT COUNT(*) AS total_weather_rows FROM weather_raw')

weather_raw total: 0 rows


,total_weather_rows
0,0


<a id='5-cleaning'></a>
## 5 · Cleaning & standardisation

Reusable functions that each accept a DataFrame and return a cleaned copy.  
These are applied when building the panel in the next step.

In [ ]:
# ── Flight cleaning ───────────────────────────────────────────────────────────

def parse_hhmm(series: pd.Series) -> pd.Series:
    """Convert BTS HHMM integer strings to 'HH:MM'; NaN for invalid."""
    def _convert(val):
        if pd.isna(val):
            return pd.NaT
        s = str(val).strip().rstrip('.0')
        if s in ('', 'nan'):
            return pd.NaT
        try:
            n = int(float(s))
        except ValueError:
            return pd.NaT
        if n == 2400:
            n = 0
        if not (0 <= n <= 2359):
            return pd.NaT
        h, m = divmod(n, 100)
        if m >= 60 or h >= 24:
            return pd.NaT
        return f'{h:02d}:{m:02d}'
    return series.map(_convert)


def clean_flights(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Normalise airport codes
    for col in ('origin_airport', 'destination_airport'):
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()
            df.loc[df[col].isin({'NAN', 'NONE', ''}), col] = pd.NA

    # flight_date → date
    if 'flight_date' in df.columns:
        df['flight_date'] = pd.to_datetime(df['flight_date'], errors='coerce').dt.date

    # HHMM time strings
    for col in ('dep_time_scheduled', 'dep_time_actual',
                'arr_time_scheduled', 'arr_time_actual'):
        if col in df.columns:
            df[col] = parse_hhmm(df[col])

    # Delay bounds
    for col in ('dep_delay', 'arr_delay'):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            bad = (df[col] < DELAY_MIN) | (df[col] > DELAY_MAX)
            df.loc[bad, col] = float('nan')

    # Cancelled flag
    if 'cancelled' in df.columns:
        df['cancelled'] = df['cancelled'].map(
            lambda x: x in (True, 1, '1', '1.0', 'True')
        ).astype(bool)

    return df


def add_dep_hour(df: pd.DataFrame) -> pd.DataFrame:
    """Extract integer departure hour from dep_time_scheduled."""
    df = df.copy()
    if 'dep_time_scheduled' in df.columns:
        df['dep_hour'] = pd.to_numeric(
            df['dep_time_scheduled'].str.extract(r'^(\d{2}):', expand=False),
            errors='coerce',
        )
    return df


# ── Weather cleaning ──────────────────────────────────────────────────────────

WEATHER_BOUNDS = {
    'temperature':   (-90.0,  60.0),
    'precipitation': (  0.0, 500.0),
    'snowfall':      (  0.0, 200.0),
    'wind_speed':    (  0.0, 400.0),
    'visibility':    (  0.0, 100.0),
    'pressure':      (870.0,1085.0),
}


def clean_weather(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # UTC timestamp
    if 'timestamp_utc' in df.columns:
        df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True, errors='coerce')

    # Drop duplicates
    dup_cols = [c for c in ('airport_iata', 'timestamp_utc') if c in df.columns]
    df = df.drop_duplicates(subset=dup_cols, keep='first')

    # Bound check
    for col, (lo, hi) in WEATHER_BOUNDS.items():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            bad = (df[col] < lo) | (df[col] > hi)
            df.loc[bad, col] = float('nan')

    return df


# ── Join validation ───────────────────────────────────────────────────────────

def validate_join(flights: pd.DataFrame, weather: pd.DataFrame) -> dict:
    fa = set(flights['origin_airport'].dropna().unique())
    wa = set(weather['airport_iata'].dropna().unique())
    stats = {
        'flight_airports':           len(fa),
        'weather_airports':          len(wa),
        'airports_in_both':          len(fa & wa),
        'airports_missing_weather':  sorted(fa - wa),
    }
    return stats


print('Cleaning functions defined')

<a id='6-panel'></a>
## 6 · Airport-hour panel dataset

Aggregates cleaned flights and weather to a single **airport × UTC-hour** row.  
Stored in `airport_hour_panel` (DuckDB) and exported as Parquet.

In [23]:
# ── Load raw tables ───────────────────────────────────────────────────────────
flights_raw = query("""
    SELECT flight_date, carrier, origin_airport, destination_airport,
           dep_time_scheduled, arr_time_scheduled, dep_delay, arr_delay, cancelled
    FROM flights_raw
""")

weather_raw = query("""
    SELECT airport_iata, timestamp_utc, temperature, precipitation,
           snowfall, wind_speed, visibility, pressure
    FROM weather_raw
""")

print(f'Loaded {len(flights_raw):,} flight rows and {len(weather_raw):,} weather rows')

# Quick join validation
print('\nJoin validation:', validate_join(flights_raw, weather_raw))

Loaded 20,334,570 flight rows and 0 weather rows

Join validation: {'flight_airports': 364, 'weather_airports': 0, 'airports_in_both': 0, 'airports_missing_weather': ['ABE', 'ABI', 'ABQ', 'ABR', 'ABY', 'ACK', 'ACT', 'ACV', 'ACY', 'ADK', 'ADQ', 'AEX', 'AGS', 'AKN', 'ALB', 'ALO', 'ALS', 'ALW', 'AMA', 'ANC', 'APN', 'ASE', 'ATL', 'ATW', 'ATY', 'AUS', 'AVL', 'AVP', 'AZA', 'AZO', 'BDL', 'BET', 'BFF', 'BFL', 'BGM', 'BGR', 'BHM', 'BIH', 'BIL', 'BIS', 'BJI', 'BKG', 'BLI', 'BLV', 'BMI', 'BNA', 'BOI', 'BOS', 'BPT', 'BQK', 'BQN', 'BRD', 'BRO', 'BRW', 'BTM', 'BTR', 'BTV', 'BUF', 'BUR', 'BWI', 'BZN', 'CAE', 'CAK', 'CDC', 'CDV', 'CGI', 'CHA', 'CHO', 'CHS', 'CID', 'CIU', 'CKB', 'CLE', 'CLL', 'CLT', 'CMH', 'CMI', 'CMX', 'CNY', 'COD', 'COS', 'COU', 'CPR', 'CRP', 'CRW', 'CSG', 'CVG', 'CWA', 'CYS', 'DAB', 'DAL', 'DAY', 'DBQ', 'DCA', 'DDC', 'DEC', 'DEN', 'DFW', 'DHN', 'DIK', 'DLG', 'DLH', 'DRO', 'DRT', 'DSM', 'DTW', 'DVL', 'EAR', 'EAT', 'EAU', 'ECP', 'EGE', 'EKO', 'ELM', 'ELP', 'ERI', 'ESC', 'EUG', 'EVV', 

In [ ]:
# Clean once and add both hour columns before any filtering
flights_clean = clean_flights(flights_raw)
flights_clean = add_dep_hour(flights_clean)
flights_clean['arr_hour'] = pd.to_numeric(
    flights_clean['arr_time_scheduled'].str.extract(r'^(\d{2}):', expand=False),
    errors='coerce',
)

# ── Departures panel ──────────────────────────────────────────────────────────
flights_dep = flights_clean.dropna(subset=['origin_airport', 'flight_date', 'dep_hour']).copy()
flights_dep['dep_hour'] = flights_dep['dep_hour'].astype(int)

dep_panel = (
    flights_dep
    .groupby(['origin_airport', 'flight_date', 'dep_hour'], dropna=True)
    .agg(
        total_departures   = ('dep_delay',  'count'),
        avg_dep_delay      = ('dep_delay',  'mean'),
        cancellation_count = ('cancelled',  'sum'),
    )
    .reset_index()
)

# ── Arrivals panel ────────────────────────────────────────────────────────────
flights_arr = flights_clean.dropna(subset=['destination_airport', 'flight_date', 'arr_hour']).copy()
flights_arr['arr_hour'] = flights_arr['arr_hour'].astype(int)

arr_panel = (
    flights_arr
    .groupby(['destination_airport', 'flight_date', 'arr_hour'], dropna=True)
    .agg(
        total_arrivals = ('arr_delay', 'count'),
        avg_arr_delay  = ('arr_delay', 'mean'),
    )
    .reset_index()
)

print(f'Departure panel: {len(dep_panel):,} rows')
print(f'Arrival panel:   {len(arr_panel):,} rows')

In [ ]:
# ── Weather panel ─────────────────────────────────────────────────────────────
wx_clean = clean_weather(weather_raw)
wx_clean = wx_clean.dropna(subset=['airport_iata', 'timestamp_utc'])
wx_clean['timestamp_utc'] = wx_clean['timestamp_utc'].dt.floor('h')

wx_panel = (
    wx_clean
    .groupby(['airport_iata', 'timestamp_utc'], dropna=True)
    .agg(
        avg_temperature   = ('temperature',   'mean'),
        avg_precipitation = ('precipitation', 'mean'),
        avg_snowfall      = ('snowfall',      'mean'),
        avg_wind_speed    = ('wind_speed',    'mean'),
        avg_visibility    = ('visibility',    'mean'),
        avg_pressure      = ('pressure',      'mean'),
    )
    .reset_index()
)

print(f'Weather panel: {len(wx_panel):,} rows')

In [ ]:
# ── Build UTC timestamps from flight date + hour ───────────────────────────────
def to_utc_timestamp(df: pd.DataFrame, date_col: str, hour_col: str) -> pd.Series:
    return pd.to_datetime(
        df[date_col].astype(str) + ' ' + df[hour_col].astype(str).str.zfill(2) + ':00:00',
        utc=True, errors='coerce'
    )

dep_panel['timestamp_utc'] = to_utc_timestamp(dep_panel, 'flight_date', 'dep_hour')
dep_panel = (
    dep_panel
    .drop(columns=['flight_date', 'dep_hour'])
    .rename(columns={'origin_airport': 'airport_iata'})
)

arr_panel['timestamp_utc'] = to_utc_timestamp(arr_panel, 'flight_date', 'arr_hour')
arr_panel = (
    arr_panel
    .drop(columns=['flight_date', 'arr_hour'])
    .rename(columns={'destination_airport': 'airport_iata'})
)

# ── Merge departures + arrivals ───────────────────────────────────────────────
flight_panel = dep_panel.merge(arr_panel, on=['airport_iata', 'timestamp_utc'], how='outer')
flight_panel['total_departures']   = flight_panel['total_departures'].fillna(0).astype(int)
flight_panel['total_arrivals']     = flight_panel['total_arrivals'].fillna(0).astype(int)
flight_panel['cancellation_count'] = flight_panel['cancellation_count'].fillna(0).astype(int)

# ── Join with weather ─────────────────────────────────────────────────────────
panel = flight_panel.merge(wx_panel, on=['airport_iata', 'timestamp_utc'], how='left')

# ── Calendar features ─────────────────────────────────────────────────────────
ts = panel['timestamp_utc']
panel['hour_of_day']      = ts.dt.hour
panel['day_of_week']      = ts.dt.dayofweek
panel['month']            = ts.dt.month
panel['year']             = ts.dt.year
panel['weekend_indicator'] = ts.dt.dayofweek >= 5

# ── Final column order ────────────────────────────────────────────────────────
COL_ORDER = [
    'airport_iata', 'timestamp_utc',
    'hour_of_day', 'day_of_week', 'month', 'year', 'weekend_indicator',
    'total_departures', 'total_arrivals', 'cancellation_count',
    'avg_dep_delay', 'avg_arr_delay',
    'avg_temperature', 'avg_precipitation', 'avg_snowfall',
    'avg_wind_speed', 'avg_visibility', 'avg_pressure',
]
for c in COL_ORDER:
    if c not in panel.columns:
        panel[c] = pd.NA
panel = panel[COL_ORDER].sort_values(['airport_iata', 'timestamp_utc']).reset_index(drop=True)

print(f'Panel shape: {panel.shape}')
panel.head(3)

In [ ]:
# ── Persist panel to DuckDB + Parquet ─────────────────────────────────────────
con = get_con()
con.execute('DELETE FROM airport_hour_panel')
con.register('panel', panel)
con.execute('INSERT INTO airport_hour_panel SELECT * FROM panel')
con.unregister('panel')
con.close()

panel_parquet = PROCESSED_DIR / 'airport_hour_panel.parquet'
write_parquet(panel, panel_parquet)

print(f'Panel saved → DuckDB and {panel_parquet}')
query('SELECT COUNT(*) AS panel_rows FROM airport_hour_panel')

<a id='7-exports'></a>
## 7 · Exports

Produces four artefacts:
1. `data/exports/airport_hour_panel.parquet` — regression-ready dataset
2. `data/exports/panel_sample.csv` — 500-row inspection sample
3. `schema/schema.sql` — SQL DDL with column documentation
4. `data/exports/data_dictionary.csv` — column-level data dictionary

In [ ]:
# ── 1 · Regression-ready Parquet ─────────────────────────────────────────────
import shutil
export_parquet = EXPORTS_DIR / 'airport_hour_panel.parquet'
shutil.copy2(panel_parquet, export_parquet)
print(f'Parquet export: {export_parquet}  ({export_parquet.stat().st_size / 1e6:.1f} MB)')

# ── 2 · CSV sample ───────────────────────────────────────────────────────────
rows_per_airport = max(1, 500 // 50)
sample_sql = f"""
    SELECT *
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY airport_iata ORDER BY timestamp_utc) AS rn
        FROM airport_hour_panel
    )
    WHERE rn <= {rows_per_airport}
    ORDER BY airport_iata, timestamp_utc
"""
sample_df = query(sample_sql)
if 'rn' in sample_df.columns:
    sample_df = sample_df.drop(columns=['rn'])

csv_path = EXPORTS_DIR / 'panel_sample.csv'
sample_df.to_csv(csv_path, index=False)
print(f'CSV sample: {csv_path}  ({len(sample_df)} rows)')

In [ ]:
# ── 3 · SQL schema ────────────────────────────────────────────────────────────
SCHEMA_SQL = """-- =========================================================================
-- Aviation Delay Research Pipeline — DuckDB Schema
-- =========================================================================

CREATE TABLE IF NOT EXISTS airports (
    iata         VARCHAR PRIMARY KEY,   -- 3-letter IATA code
    icao         VARCHAR,               -- 4-letter ICAO code
    airport_name VARCHAR,
    city         VARCHAR,
    state        VARCHAR,               -- 2-letter US state / territory
    latitude     DOUBLE,               -- decimal degrees N
    longitude    DOUBLE,               -- decimal degrees E (negative = W)
    timezone     VARCHAR                -- IANA timezone string
);

CREATE TABLE IF NOT EXISTS flights_raw (
    flight_date          DATE,          -- local departure date
    carrier              VARCHAR,       -- IATA airline code
    flight_number        VARCHAR,
    tail_number          VARCHAR,       -- FAA N-number; may be NULL
    origin_airport       VARCHAR,
    destination_airport  VARCHAR,
    dep_time_scheduled   VARCHAR,       -- HH:MM local
    dep_time_actual      VARCHAR,
    dep_delay            DOUBLE,        -- minutes; positive = late
    arr_time_scheduled   VARCHAR,
    arr_time_actual      VARCHAR,
    arr_delay            DOUBLE,
    cancelled            BOOLEAN,
    cancellation_code    VARCHAR,       -- A=carrier B=weather C=NAS D=security
    source_file          VARCHAR        -- provenance tag e.g. BTS_2022_01
);

CREATE TABLE IF NOT EXISTS weather_raw (
    airport_iata  VARCHAR,
    station_id    VARCHAR,              -- Meteostat station ID
    timestamp_utc TIMESTAMPTZ,
    temperature   DOUBLE,              -- °C
    precipitation DOUBLE,              -- mm/hr
    snowfall      DOUBLE,              -- mm/hr
    wind_speed    DOUBLE,              -- km/h
    visibility    DOUBLE,              -- km
    pressure      DOUBLE               -- hPa
);

CREATE TABLE IF NOT EXISTS airport_hour_panel (
    airport_iata       VARCHAR,
    timestamp_utc      TIMESTAMPTZ,    -- UTC hour (floored)
    hour_of_day        INTEGER,        -- 0-23
    day_of_week        INTEGER,        -- 0=Mon 6=Sun
    month              INTEGER,        -- 1-12
    year               INTEGER,
    weekend_indicator  BOOLEAN,
    total_departures   INTEGER,
    total_arrivals     INTEGER,
    cancellation_count INTEGER,
    avg_dep_delay      DOUBLE,
    avg_arr_delay      DOUBLE,
    avg_temperature    DOUBLE,
    avg_precipitation  DOUBLE,
    avg_snowfall       DOUBLE,
    avg_wind_speed     DOUBLE,
    avg_visibility     DOUBLE,
    avg_pressure       DOUBLE,
    PRIMARY KEY (airport_iata, timestamp_utc)
);
"""

schema_path = SCHEMA_DIR / 'schema.sql'
schema_path.write_text(SCHEMA_SQL, encoding='utf-8')
print(f'Schema SQL: {schema_path}')

In [ ]:
# ── 4 · Data dictionary ───────────────────────────────────────────────────────
DATA_DICT_ROWS = [
    # airports
    ('airports','iata',         'VARCHAR',    '3-letter IATA airport code (PK)'),
    ('airports','icao',         'VARCHAR',    '4-letter ICAO airport code'),
    ('airports','airport_name', 'VARCHAR',    'Full official airport name'),
    ('airports','city',         'VARCHAR',    'Nearest city'),
    ('airports','state',        'VARCHAR',    '2-letter US state or territory'),
    ('airports','latitude',     'DOUBLE',     'Decimal degrees north'),
    ('airports','longitude',    'DOUBLE',     'Decimal degrees east (negative = west)'),
    ('airports','timezone',     'VARCHAR',    'IANA timezone string'),
    # flights_raw
    ('flights_raw','flight_date',         'DATE',    'Local calendar date of scheduled departure'),
    ('flights_raw','carrier',             'VARCHAR', '2-letter IATA airline code'),
    ('flights_raw','flight_number',       'VARCHAR', 'Marketing flight number'),
    ('flights_raw','tail_number',         'VARCHAR', 'FAA aircraft tail number; may be NULL'),
    ('flights_raw','origin_airport',      'VARCHAR', 'IATA departure airport code'),
    ('flights_raw','destination_airport', 'VARCHAR', 'IATA arrival airport code'),
    ('flights_raw','dep_time_scheduled',  'VARCHAR', 'Scheduled local departure HH:MM'),
    ('flights_raw','dep_time_actual',     'VARCHAR', 'Actual local departure HH:MM; NULL if cancelled'),
    ('flights_raw','dep_delay',           'DOUBLE',  'Departure delay in minutes (+ = late)'),
    ('flights_raw','arr_time_scheduled',  'VARCHAR', 'Scheduled local arrival HH:MM'),
    ('flights_raw','arr_time_actual',     'VARCHAR', 'Actual local arrival HH:MM'),
    ('flights_raw','arr_delay',           'DOUBLE',  'Arrival delay in minutes'),
    ('flights_raw','cancelled',           'BOOLEAN', 'True if flight was cancelled'),
    ('flights_raw','cancellation_code',   'VARCHAR', 'A=Carrier B=Weather C=NAS D=Security'),
    ('flights_raw','source_file',         'VARCHAR', 'Provenance tag e.g. BTS_2022_01'),
    # weather_raw
    ('weather_raw','airport_iata',  'VARCHAR',     'IATA code of associated airport'),
    ('weather_raw','station_id',    'VARCHAR',     'Meteostat station identifier'),
    ('weather_raw','timestamp_utc', 'TIMESTAMPTZ', 'Observation time in UTC'),
    ('weather_raw','temperature',   'DOUBLE',      'Air temperature (°C)'),
    ('weather_raw','precipitation', 'DOUBLE',      'Precipitation (mm/hr)'),
    ('weather_raw','snowfall',      'DOUBLE',      'Snowfall (mm/hr)'),
    ('weather_raw','wind_speed',    'DOUBLE',      'Mean wind speed (km/h)'),
    ('weather_raw','visibility',    'DOUBLE',      'Horizontal visibility (km)'),
    ('weather_raw','pressure',      'DOUBLE',      'Sea-level pressure (hPa)'),
    # airport_hour_panel
    ('airport_hour_panel','airport_iata',       'VARCHAR',     'IATA airport code (join key)'),
    ('airport_hour_panel','timestamp_utc',      'TIMESTAMPTZ', 'UTC hour floored to hour boundary — join key'),
    ('airport_hour_panel','hour_of_day',        'INTEGER',     '0–23'),
    ('airport_hour_panel','day_of_week',        'INTEGER',     '0=Monday … 6=Sunday'),
    ('airport_hour_panel','month',              'INTEGER',     '1–12'),
    ('airport_hour_panel','year',               'INTEGER',     '2022–2024'),
    ('airport_hour_panel','weekend_indicator',  'BOOLEAN',     'True if Saturday or Sunday'),
    ('airport_hour_panel','total_departures',   'INTEGER',     'Departing flights in this airport-hour'),
    ('airport_hour_panel','total_arrivals',     'INTEGER',     'Arriving flights in this airport-hour'),
    ('airport_hour_panel','cancellation_count', 'INTEGER',     'Cancelled departures in this airport-hour'),
    ('airport_hour_panel','avg_dep_delay',      'DOUBLE',      'Mean departure delay (minutes)'),
    ('airport_hour_panel','avg_arr_delay',      'DOUBLE',      'Mean arrival delay (minutes)'),
    ('airport_hour_panel','avg_temperature',    'DOUBLE',      'Mean temperature (°C)'),
    ('airport_hour_panel','avg_precipitation',  'DOUBLE',      'Mean precipitation (mm/hr)'),
    ('airport_hour_panel','avg_snowfall',       'DOUBLE',      'Mean snowfall (mm/hr)'),
    ('airport_hour_panel','avg_wind_speed',     'DOUBLE',      'Mean wind speed (km/h)'),
    ('airport_hour_panel','avg_visibility',     'DOUBLE',      'Mean visibility (km)'),
    ('airport_hour_panel','avg_pressure',       'DOUBLE',      'Mean pressure (hPa)'),
]

dict_df = pd.DataFrame(DATA_DICT_ROWS, columns=['table','column','dtype','description'])
dict_path = EXPORTS_DIR / 'data_dictionary.csv'
dict_df.to_csv(dict_path, index=False)
print(f'Data dictionary: {dict_path}  ({len(dict_df)} entries)')
dict_df.head(8)

<a id='8-inspect'></a>
## 8 · Quick data inspection

Sanity-check queries on the completed database — no analysis, just coverage and shape checks.

In [ ]:
# Row counts across all tables
counts = query("""
    SELECT 'airports'          AS table_name, COUNT(*) AS rows FROM airports
    UNION ALL
    SELECT 'flights_raw',                     COUNT(*) FROM flights_raw
    UNION ALL
    SELECT 'weather_raw',                     COUNT(*) FROM weather_raw
    UNION ALL
    SELECT 'airport_hour_panel',              COUNT(*) FROM airport_hour_panel
""")
counts

In [ ]:
# Panel date coverage per airport (first & last hour, total rows)
query("""
    SELECT
        airport_iata,
        MIN(timestamp_utc)  AS first_hour,
        MAX(timestamp_utc)  AS last_hour,
        COUNT(*)            AS panel_rows,
        SUM(total_departures) AS total_dep
    FROM airport_hour_panel
    GROUP BY airport_iata
    ORDER BY total_dep DESC
    LIMIT 15
""")

In [ ]:
# Hourly delay profile across all airports
query("""
    SELECT
        hour_of_day,
        ROUND(AVG(avg_dep_delay), 2)  AS mean_dep_delay_min,
        ROUND(AVG(avg_arr_delay), 2)  AS mean_arr_delay_min,
        SUM(total_departures)          AS total_departures
    FROM airport_hour_panel
    WHERE avg_dep_delay IS NOT NULL
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""")

In [ ]:
# Weather completeness: fraction of panel rows with temperature data
query("""
    SELECT
        airport_iata,
        COUNT(*)                          AS total_hours,
        COUNT(avg_temperature)            AS hours_with_temp,
        ROUND(100.0 * COUNT(avg_temperature) / COUNT(*), 1) AS pct_coverage
    FROM airport_hour_panel
    GROUP BY airport_iata
    ORDER BY pct_coverage ASC
    LIMIT 10
""")

In [ ]:
# Cancellation summary by month-year
query("""
    SELECT
        year,
        month,
        SUM(cancellation_count) AS total_cancellations,
        SUM(total_departures)   AS total_departures,
        ROUND(100.0 * SUM(cancellation_count) / NULLIF(SUM(total_departures), 0), 2) AS cancel_rate_pct
    FROM airport_hour_panel
    GROUP BY year, month
    ORDER BY year, month
""")

<a id="9-stats"></a>
## 9 · Descriptive statistics

Per-column summary of the final `airport_hour_panel` dataset: coverage rate, central tendency, spread, and range.

In [ ]:
panel = query("SELECT * FROM airport_hour_panel")

numeric_cols = [
    'total_departures', 'total_arrivals', 'cancellation_count',
    'avg_dep_delay', 'avg_arr_delay',
    'avg_temperature', 'avg_precipitation', 'avg_snowfall',
    'avg_wind_speed', 'avg_visibility', 'avg_pressure',
]

n = len(panel)
rows = []
for col in numeric_cols:
    s = panel[col]
    non_null = s.notna().sum()
    rows.append({
        'column':       col,
        'non_null':     non_null,
        'coverage_%':   round(100 * non_null / n, 1),
        'mean':         round(s.mean(),   3),
        'std':          round(s.std(),    3),
        'min':          round(s.min(),    3),
        'p25':          round(s.quantile(0.25), 3),
        'median':       round(s.median(), 3),
        'p75':          round(s.quantile(0.75), 3),
        'max':          round(s.max(),    3),
    })

desc = pd.DataFrame(rows).set_index("column")

print(f"Rows : {n:,}")
print(f"Airports : {panel['airport_iata'].nunique()} unique")
print(f"Period   : {panel['timestamp_utc'].min()} → {panel['timestamp_utc'].max()}")
print(f"Weekend  : {panel['weekend_indicator'].sum():,} rows ({100*panel['weekend_indicator'].mean():.1f}%)")
print()
desc

In [ ]:
# Final export listing
print('=== Pipeline complete ===')
print(f'Database : {DB_PATH}')
print()
print('Exports:')
for f in sorted(EXPORTS_DIR.glob('*')):
    print(f'  {f.name:<40} {f.stat().st_size / 1e3:8.1f} KB')
print()
print('Schema:')
for f in sorted(SCHEMA_DIR.glob('*')):
    print(f'  {f.name}')